In [1]:
# Here I am importing pandas, numpy, re and os, and creating the output folders.
import pandas as pd
import numpy as np
import re
import os

os.makedirs('01_interim', exist_ok=True)
os.makedirs('02_processed', exist_ok=True)

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
## Part A — KTAS triage dataset

In [3]:
# Here I am loading the raw KTAS file
ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')

print('KTAS loaded successfully!')
print('Shape:', ktas.shape)
print('Columns:', ktas.columns.tolist())

KTAS loaded successfully!
Shape: (1267, 24)
Columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_RN', 'Diagnosis in ED', 'Disposition', 'KTAS_expert', 'Error_group', 'Length of stay_min', 'KTAS duration_min', 'mistriage']


In [4]:
# Here I am profiling the raw KTAS data and checking for missing values
print('=== DATA TYPES ===')
print(ktas.dtypes)

print('\n=== MISSING VALUES ===')
print(ktas.isnull().sum())

=== DATA TYPES ===
Group                       int64
Sex                         int64
Age                         int64
Patients number per hour    int64
Arrival mode                int64
Injury                      int64
Chief_complain                str
Mental                      int64
Pain                        int64
NRS_pain                      str
SBP                           str
DBP                           str
HR                            str
RR                            str
BT                            str
Saturation                    str
KTAS_RN                     int64
Diagnosis in ED               str
Disposition                 int64
KTAS_expert                 int64
Error_group                 int64
Length of stay_min          int64
KTAS duration_min             str
mistriage                   int64
dtype: object

=== MISSING VALUES ===
Group                         0
Sex                           0
Age                           0
Patients number per hour      0

In [5]:
# Here I am profiling the raw KTAS data.
print('=== DATA TYPES ===')
print(ktas.dtypes)

print('\n=== MISSING VALUES BEFORE CLEANING ===')
print(ktas.isnull().sum()[ktas.isnull().sum() > 0])

=== DATA TYPES ===
Group                       int64
Sex                         int64
Age                         int64
Patients number per hour    int64
Arrival mode                int64
Injury                      int64
Chief_complain                str
Mental                      int64
Pain                        int64
NRS_pain                      str
SBP                           str
DBP                           str
HR                            str
RR                            str
BT                            str
Saturation                    str
KTAS_RN                     int64
Diagnosis in ED               str
Disposition                 int64
KTAS_expert                 int64
Error_group                 int64
Length of stay_min          int64
KTAS duration_min             str
mistriage                   int64
dtype: object

=== MISSING VALUES BEFORE CLEANING ===
Saturation         688
Diagnosis in ED      2
dtype: int64


In [6]:
# Here I am checking for duplicate records.
n_before = len(ktas)
dupes = ktas.duplicated().sum()
print(f'Duplicate rows found: {dupes}')

if dupes > 0:
    ktas = ktas.drop_duplicates()
    print(f'Duplicates removed. Rows: {n_before} -> {len(ktas)}')
else:
    print('No duplicate records found — nothing removed.')

Duplicate rows found: 0
No duplicate records found — nothing removed.


In [7]:
# Here I am converting the clinical columns to numeric (garbage text becomes NaN).
numeric_columns_to_fix = ['NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'Age']
for col in numeric_columns_to_fix:
    if col in ktas.columns:
        ktas[col] = pd.to_numeric(ktas[col], errors='coerce')

print('Data types after conversion:')
print(ktas[numeric_columns_to_fix].dtypes)

Data types after conversion:
NRS_pain      float64
SBP           float64
DBP           float64
HR            float64
RR            float64
BT            float64
Saturation    float64
Age             int64
dtype: object


In [8]:
# Here I am checking for inconsistent records (no pain but score > 0).
ktas['Pain'] = pd.to_numeric(ktas['Pain'], errors='coerce')
contradictions = ktas[(ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0)]
print(f'Contradictory rows found: {len(contradictions)}')

ktas = ktas[~((ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0))]
print(f'Rows remaining: {len(ktas)}')

Contradictory rows found: 0
Rows remaining: 1267


In [9]:
# Here I am handling missing values for EVERY column dynamically.
# Numeric columns get the median; text/categorical columns get the most frequent value (mode).
print('Imputing missing values across ALL columns:')

for col in ktas.columns:
    # Only impute if there are actually missing values
    if ktas[col].isnull().sum() > 0:
        n_missing = ktas[col].isnull().sum()
        
        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(ktas[col]):
            med = ktas[col].median()
            ktas[col] = ktas[col].fillna(med)
            print(f"  '{col}': filled {n_missing} nulls with median ({med})")
        else:
            # For text/categorical columns
            mode_val = ktas[col].mode()[0] if not ktas[col].mode().empty else 'Unknown'
            ktas[col] = ktas[col].fillna(mode_val)
            print(f"  '{col}': filled {n_missing} nulls with mode ({mode_val})")

print('\nImputation complete. Any remaining nulls?', ktas.isnull().sum().sum())

Imputing missing values across ALL columns:
  'NRS_pain': filled 556 nulls with median (4.0)
  'SBP': filled 25 nulls with median (130.0)
  'DBP': filled 29 nulls with median (80.0)
  'HR': filled 20 nulls with median (82.0)
  'RR': filled 22 nulls with median (20.0)
  'BT': filled 18 nulls with median (36.5)
  'Saturation': filled 697 nulls with median (98.0)
  'Diagnosis in ED': filled 2 nulls with mode (Acute gastroenteritis)

Imputation complete. Any remaining nulls? 0


In [10]:
# Here I am encoding the categorical variables into readable labels.
ktas['Sex'] = ktas['Sex'].map({1: 'Female', 2: 'Male'})
ktas['Mental'] = ktas['Mental'].map({1: 'Alert', 2: 'Verbal', 3: 'Pain', 4: 'Unresponsive'})
ktas['Injury'] = ktas['Injury'].map({1: 'No', 2: 'Yes'})
ktas['Pain'] = ktas['Pain'].map({0: 'No', 1: 'Yes'})
ktas['Arrival mode'] = ktas['Arrival mode'].replace({5: 'Other', 6: 'Other', 7: 'Other'})

print('Categories mapped.')

Categories mapped.


In [11]:
# Here I am cleaning the free-text Chief_complain column.
abbreviation_map = {
    r'\brt\.?\b': 'right', r'\blt\.?\b': 'left', r'\bbilat\.?\b': 'bilateral',
    r'\bant\.?\b': 'anterior', r'\bpost\.?\b': 'posterior', r'\bext\.?\b': 'extremity',
    r'\bperi-?umbilical\b': 'periumbilical', r'\babd\.?\b': 'abdominal', r'\bHA\b': 'headache',
    r'\bH-\b': 'headache ', r'\bLBP\b': 'low back pain', r'\bCP\b': 'chest pain',
    r'\bSOB\b': 'shortness of breath', r'\bN/V\b': 'nausea and vomiting', r'\bF/C\b': 'fever and chills',
    r'\bC/S\b': 'chills and sweating', r'\bDZ\b': 'dizziness', r'\bLOC\b': 'loss of consciousness',
    r'\bFB\b': 'foreign body', r'\bTMJ\b': 'temporomandibular joint', r'\bPTx\b': 'pneumothorax',
    r'\bAVF\b': 'arteriovenous fistula', r'\bPPROM\b': 'preterm premature rupture of membranes',
    r'\bBST\b': 'blood sugar test', r'\bBSL\b': 'blood sugar level', r'\bHb\b': 'hemoglobin',
    r'\bDI\b': 'drug intoxication', r'\bRLQ\b': 'right lower quadrant', r'\bLLQ\b': 'left lower quadrant',
    r'\bRUQ\b': 'right upper quadrant', r'\bLUQ\b': 'left upper quadrant', r'\bO\.?T\.?\b': 'operating theatre',
    r'\bICU\b': 'intensive care unit', r'\bED\b': 'emergency department', r'\bGI\b': 'gastrointestinal',
    r'\bGU\b': 'genitourinary', r'\bCVA\b': 'cerebrovascular accident', r'\bCNS\b': 'central nervous system',
    r'\bBT\b': 'body temperature', r'\bBP\b': 'blood pressure', r'\bSBP\b': 'systolic blood pressure',
    r'\bDBP\b': 'diastolic blood pressure', r'\bHR\b': 'heart rate', r'\bRR\b': 'respiratory rate',
    r'\bSpO2\b': 'oxygen saturation', r'\bNRS\b': 'numeric rating scale', r'\bPalp\.?\b': 'palpitation',
    r'\bSz\b': 'seizure', r'\bCONV\b': 'convulsion', r'\bAMS\b': 'altered mental status',
    r'\bGTC\b': 'generalised tonic-clonic seizure',
}

def clean_text(text):
    if pd.isna(text): return 'unknown'
    text = str(text).strip()
    text = re.sub(r'\?+', '', text)
    text = re.sub(r'[/(),+.:]', ' ', text)
    text = text.replace('-', ' ')
    for abbr, full in abbreviation_map.items():
        text = re.sub(abbr, full, text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text if text else 'unknown'

ktas['Chief_complain_cleaned'] = ktas['Chief_complain'].apply(clean_text)
print("Cleaned column created.")

Cleaned column created.


In [12]:
# Here I am saving the cleaned KTAS, keeping ALL columns exactly as they were.
ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)
print(f'Saved 01_interim/ktas_cleaned.csv  (rows={ktas.shape[0]}, cols={ktas.shape[1]})')

# Verify file saved and has no missing values
ktas_check = pd.read_csv('01_interim/ktas_cleaned.csv')
print(f'Total missing cells in saved file: {ktas_check.isnull().sum().sum()}')
assert ktas_check.isnull().sum().sum() == 0, 'Error: Missing values still exist!'
print('Verification passed. 0 missing values.')

Saved 01_interim/ktas_cleaned.csv  (rows=1267, cols=25)
Total missing cells in saved file: 0
Verification passed. 0 missing values.


In [13]:
## Part B — Patient Flow dataset

In [14]:
# Here I am loading the raw Patient Flow simulation file.
flow = pd.read_csv('00_raw/patient_flow_original.csv', encoding='latin-1')

print('Patient Flow loaded successfully!')
print('Shape:', flow.shape)
print('Missing cells:', flow.isnull().sum().sum())

Patient Flow loaded successfully!
Shape: (60000, 14)
Missing cells: 0


In [15]:
# Here I am checking for duplicates and renaming columns.
dupes = flow.duplicated().sum()
if dupes > 0:
    flow = flow.drop_duplicates()
    print(f'Removed {dupes} duplicates.')

flow = flow.rename(columns={
    'ArrivalTime': 'arrival_min', 'DocWaitTime': 'doc_wait_min',
    'MedWaitTime': 'med_wait_min', 'DocServiceTime': 'doc_service_min',
    'MedServiceTime': 'med_service_min', 'TotalTimeInHospital': 'total_time_min',
    'Priority': 'priority_level', 'NumDoctors': 'num_doctors',
})
print('Columns renamed.')

Columns renamed.


In [16]:
# Here I am deriving summary timing columns and converting minutes to datetime.
flow['total_wait_min'] = flow['doc_wait_min'] + flow['med_wait_min']
flow['total_service_min'] = flow['doc_service_min'] + flow['med_service_min']
flow['patient_final_time_min'] = flow['total_wait_min'] + flow['total_service_min']

# Convert to real datetime
flow['DateTime'] = pd.to_datetime('2024-01-01 08:00:00') + pd.to_timedelta(flow['arrival_min'], unit='m')

# Save cleaned flow
flow.to_csv('01_interim/patient_flow_cleaned.csv', index=False)
print(f'Saved 01_interim/patient_flow_cleaned.csv (rows={flow.shape[0]})')

Saved 01_interim/patient_flow_cleaned.csv (rows=60000)


In [17]:
## Part C — Hourly time series per scenario

In [18]:
# Here I am aggregating Patient Flow per scenario per hour.
flow_hourly = (
    flow.groupby(['Scenario', pd.Grouper(key='DateTime', freq='h')])
        .agg(
            patient_count=('PatientID', 'count'),
            avg_total_wait=('total_wait_min', 'mean'),
            avg_total_service=('total_service_min', 'mean'),
            avg_patient_final_time=('patient_final_time_min', 'mean'),
            max_patient_final_time=('patient_final_time_min', 'max'),
        )
        .reset_index()
)

flow_hourly.to_csv('02_processed/patient_flow_timeseries.csv', index=False)
print(f'Saved 02_processed/patient_flow_timeseries.csv (rows={flow_hourly.shape[0]})')

Saved 02_processed/patient_flow_timeseries.csv (rows=111)


In [19]:
## Part D — Final verification

In [21]:
# Final check on saved files.
ktas_final = pd.read_csv('01_interim/ktas_cleaned.csv')
flow_final = pd.read_csv('01_interim/patient_flow_cleaned.csv')
ts_final   = pd.read_csv('02_processed/patient_flow_timeseries.csv')

print(f'KTAS cleaned: {ktas_final.shape}')
print(f'Flow cleaned: {flow_final.shape}')
print(f'Time series : {ts_final.shape}')


KTAS cleaned: (1267, 25)
Flow cleaned: (60000, 18)
Time series : (111, 7)
